In [1]:
!nvidia-smi

Sun Nov 16 20:11:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:AC:00.0 Off |                  Off |
| 30%   39C    P5             77W /  300W |     430MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

max_seq_length = 2048
dtype = None

model_name = "unsloth/Qwen3-32B"
new_model_id = "Qwen3-32B-pyteal-desc" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
  )


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-11-16 20:11:56.129378: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-16 20:11:56.142799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763323916.158699  291018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763323916.163735  291018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763323916.177188  291018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
Switching to PyTorch attention since your Xformers is broken.

/home/habes/anaconda3/envs/llmenv/lib/python3.10/site-packages/flash_attn_2_cuda.cpython-310-x86_64-linux-gnu.so: undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationESs
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.12: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.422 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unslo

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth 2025.10.12 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


In [3]:
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

if "QwQ" in model_name or "Qwen" in model_name:
    #Chat template for Qwen model
    tokenizer.chat_template = """
{% for message in messages %}
{% if message['role'] == 'developer' or message['role'] == 'system' %}
<|im_start|>system
{{ message['content'] }}<|im_end|>\n

{% elif message['role'] == 'user' %}
<|im_start|>user{{ message['content'] }}<|im_end|>user\n
{% elif message['role'] == 'assistant' %}
{% if message['description'] %}
<|im_start|>assistant
<final>
{{ message['description'] }}
</final>
<|im_end|>\n
{% else %}
<|im_start|>assistant
<think>
{{ message['thinking'] }}</think>\n
<final>
{{ message['content'] }}
</final><|im_end|>\n
{% endif %}
{% endif %}
{% endfor %}

{%- if add_generation_prompt %}
{%- if think %}
<|im_start|>assistant
<think>
{%- else %}
<|im_start|>assistant
<final>
{%- endif %}
{%- endif %}

    """

    training_kwargs = dict(instruction_part = "<|im_start|>user", response_part="<|im_end|>user")

def formatting_prompts_func(examples):
  convos = examples["messages"]
  texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
  return { "text" : texts, }

dataset_train = load_dataset("json", data_files="datasets/dataset_think_format.jsonl", split="train").shuffle(seed=43)

dataset_eval = load_dataset("json", data_files="datasets/validation_dataset.jsonl", split="train").shuffle(seed=43)

dataset_train = standardize_sharegpt(dataset_train)
dataset_train = dataset_train.map(formatting_prompts_func, batched = True,)

dataset_eval = standardize_sharegpt(dataset_eval)
dataset_eval = dataset_eval.map(formatting_prompts_func, batched = True,)

print(dataset_eval)
print(dataset_train)

print(dataset_train[2]['text'])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'messages', 'text'],
    num_rows: 25
})
Dataset({
    features: ['id', 'messages', 'text'],
    num_rows: 500
})

<|im_start|>system

You are an expert smart contract security auditor specialized in the Algorand blockchain and PyTeal. 
Your task is to analyze PyTeal code precisely and systematically to identify security vulnerabilities.

### Required behavior:
- Always perform a structured, explicit reasoning phase first and put it inside the <think> block.
  - In <think>...</think> you must:
    - Summarize the contract's purpose and high-level architecture.
    - Inspect the logic block-by-block (or line-by-line for short snippets).
    - Note any suspicious patterns, missing authorization checks, unsafe transaction handling, or other issues with evidence (point to the code lines/constructs).
    - Use concise, technical language and show the chain of reasoning (why you suspect an issue).

- After </think>, provide the final judgment in the <final>...<

In [4]:
from trl import SFTConfig

sft_config = SFTConfig(
    # === Efficienza GPU ===
    per_device_train_batch_size = 1,     
    gradient_accumulation_steps = 8,     
    gradient_checkpointing = True,       

    # === Durata training ===
    num_train_epochs = 3,                
    max_seq_length = 2048,               
    warmup_ratio = 0.05,                 
    lr_scheduler_type = "cosine",        

    # === Ottimizzazione ===
    learning_rate = 5e-5,              
    weight_decay = 0.02,
    optim = "paged_adamw_8bit",          
    max_grad_norm = 1.0,                

    # === Evaluation & logging ===
    eval_strategy = "steps",
    eval_steps = 10,                     
    logging_steps = 10,
    save_strategy = "epoch",
    save_total_limit = 5,        # tiene solo gli ultimi 3 checkpoint       

    # === Reproducibilità ===
    seed = 3407,

    # === Output & tracking ===
    output_dir = f"models/{new_model_id}",
    report_to = "none",                  
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_eval,
    args = sft_config
)


training_kwargs = dict(instruction_part = "<|im_start|>user", response_part="<|im_end|>user")

trainer = train_on_responses_only(
    trainer,
    **training_kwargs,
)

tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/500 [00:00<?, ? examples/s]

num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.
[datasets.arrow_dataset|WARNING]num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.


Unsloth: Tokenizing ["text"] (num_proc=25):   0%|          | 0/25 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/500 [00:00<?, ? examples/s]

num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.
[datasets.arrow_dataset|WARNING]num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.


Map (num_proc=25):   0%|          | 0/25 [00:00<?, ? examples/s]

"                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             \n\n<|im_start|>assistant\n<think>\n\n\nI am analyzing this PyTeal code for a Hashed Timelock Contract (HTLC) implemented as a smart signature. First, I need to understand its core purpose: it's a stateless contract that facilitates atomic swaps by allowing a seller to claim funds with a secret preimage or a buyer to reclaim them after a timeout.\n\nLet me break down the security logic step by step. The contract starts wit

In [5]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[5]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [6]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 536,870,912 of 33,298,994,176 (1.61% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.323400,0.802606
20,0.955100,0.662995
30,0.843000,0.615756
40,0.836200,0.609549
50,0.761000,0.598417
60,0.739400,0.585406
70,0.692300,0.577736
80,0.632100,0.581082
90,0.592100,0.565557
100,0.642500,0.568852


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
# === SALVATAGGIO MODELLO NELLA REPO HF ===
import os
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import HfApi, create_repo

# Configurazione
TOKEN = os.environ["HF_TOKEN"]
USERNAME = "Nikix999"

# Definizione dei modelli da caricare
models_to_upload = [
    {
        "local_path": "./models/Qwen3-32B-pyteal-desc/checkpoint-189", # Percorso locale Qwen
        "repo_name": "Qwen3-32B-PyTeal-Audit",
        "private": False 
    },
    {
        "local_path": "./models/QwQ-32B-pyteal-desc/checkpoint-189", # Percorso locale QwQ
        "repo_name": "QwQ-32B-PyTeal-Audit",
        "private": False
    }
]

api = HfApi(token=TOKEN)

for model in models_to_upload:
    full_repo_id = f"{USERNAME}/{model['repo_name']}"
    
    print(f"Creating repo: {full_repo_id}...")
    try:
        create_repo(full_repo_id, repo_type="model", private=model['private'], token=TOKEN)
    except Exception as e:
        print(f"Repo already exists or error: {e}")

    print(f"Uploading {model['local_path']} to {full_repo_id}...")
    api.upload_folder(
        folder_path=model['local_path'],
        repo_id=full_repo_id,
        repo_type="model",
        commit_message=f"Upload fine-tuned checkpoint for {model['repo_name']}"
    )
    print("Done!")